In [10]:
# Sam Brown
# sam_brown
# sam_brown@mines.edu
# Goal: Create data pipeline to retrieve seismic data for whillans events

import os
import obspy
from obspy.clients.fdsn.mass_downloader import Restrictions, MassDownloader, GlobalDomain

seismogram_path = './data/seismograms' # Set download path
xml_path = './data/stations' # Set XML path

In [12]:
# Manually set start time, number of days of data to download, and station info
start_time = obspy.UTCDateTime(2012, 1, 1)
days_to_download = 365
network = "IU" 
station = "QSPA" # South Pole seismic station
location = "00"
channel = "BH?" # ? is a wildcard for all channels starting with BH

# Set up to download from IRIS, using global domain because we're specifying
# specific stations
mdl = MassDownloader(providers=['IRIS'])
domain = GlobalDomain()

SEC_PER_DAY = 24 * 3600

# Download four days of data in one day chunks
for day in range(days_to_download):
    end_time = start_time + SEC_PER_DAY
    restrictions = Restrictions(
        starttime=start_time,
        endtime=end_time,
        chunklength_in_sec=SEC_PER_DAY, #1 day chunks
        network=network,
        station=station,
        location=location,
        channel=channel,
        reject_channels_with_gaps=False,
        minimum_length=0.0,
        minimum_interstation_distance_in_m=100.0 # Guard against same station having different names
        )
    mdl.download(domain, restrictions, mseed_storage=seismogram_path,
                stationxml_storage=xml_path,)
   
    start_time += SEC_PER_DAY # Update to next day

[2025-06-23 12:22:56,650] - obspy.clients.fdsn.mass_downloader - INFO: Initializing FDSN client(s) for IRIS.
[2025-06-23 12:22:56,650] - obspy.clients.fdsn.mass_downloader - INFO: Initializing FDSN client(s) for IRIS.
[2025-06-23 12:22:56,668] - obspy.clients.fdsn.mass_downloader - INFO: Successfully initialized 1 client(s): IRIS.
[2025-06-23 12:22:56,668] - obspy.clients.fdsn.mass_downloader - INFO: Successfully initialized 1 client(s): IRIS.
[2025-06-23 12:22:56,669] - obspy.clients.fdsn.mass_downloader - INFO: Total acquired or preexisting stations: 0
[2025-06-23 12:22:56,669] - obspy.clients.fdsn.mass_downloader - INFO: Total acquired or preexisting stations: 0
[2025-06-23 12:22:56,670] - obspy.clients.fdsn.mass_downloader - INFO: Client 'IRIS' - Requesting reliable availability.
[2025-06-23 12:22:56,670] - obspy.clients.fdsn.mass_downloader - INFO: Client 'IRIS' - Requesting reliable availability.
[2025-06-23 12:22:56,921] - obspy.clients.fdsn.mass_downloader - INFO: Client 'IRIS'

KeyboardInterrupt: 

[2025-06-23 12:30:41,949] - obspy.clients.fdsn.mass_downloader - INFO: Client 'IRIS' - Successfully downloaded 3 channels (of 3)
[2025-06-23 12:30:41,949] - obspy.clients.fdsn.mass_downloader - INFO: Client 'IRIS' - Successfully downloaded 3 channels (of 3)


In [35]:
# Load seismograms
inv =  obspy.read_inventory("./data/stations/*")
stream = obspy.Stream()

for trace_index in range(days_to_download):
    print(f"Working on {trace_index}")
    st_index = trace_index * 3  # Each day has 3 channels
    ed_index = trace_index * 3 + 3 
    for file in os.listdir(seismogram_path)[st_index:ed_index]:
        # Each mseed file may contain multiple traces that need to be combined
        temp_stream = obspy.read(f'{seismogram_path}/{file}')
        temp_stream.attach_response(inv)
        t1 = temp_stream[0]
    
        for trace in temp_stream[1:]:
            t1 = t1.__add__(trace, method=1,fill_value=0)
        stream.append(t1)

# Seperate into North, South, and Vertical components
stream.sort()
N = stream.select(channel='BH1')
E = stream.select(channel='BH2')
Z = stream.select(channel='BHZ')

# Make daily streams
day_streams = []
for day in range(len(N)):
    day_stream = obspy.Stream(traces=[N[day],E[day],Z[day]])
    day_streams.append(day_stream)


Working on 0
Working on 1
Working on 2
Working on 3
Working on 4
Working on 5
Working on 6
Working on 7
Working on 8
Working on 9
Working on 10
Working on 11
Working on 12
Working on 13
Working on 14
Working on 15
Working on 16
Working on 17
Working on 18
Working on 19
Working on 20
Working on 21
Working on 22
Working on 23
Working on 24
Working on 25
Working on 26
Working on 27
Working on 28
Working on 29
Working on 30
Working on 31
Working on 32
Working on 33
Working on 34
Working on 35
Working on 36
Working on 37
Working on 38
Working on 39
Working on 40
Working on 41
Working on 42
Working on 43
Working on 44
Working on 45
Working on 46
Working on 47
Working on 48
Working on 49
Working on 50
Working on 51
Working on 52
Working on 53
Working on 54
Working on 55
Working on 56
Working on 57
Working on 58
Working on 59
Working on 60
Working on 61
Working on 62
Working on 63
Working on 64
Working on 65
Working on 66
Working on 67
Working on 68
Working on 69
Working on 70
Working on 71
Wo

In [ ]:
# Now we will pull all of the start times for the events in 2012 and generate streams for each event.